# 02 단일 이미지 객체검출

공공데이터 기반 샘플 이미지에 대해 YOLOv8 사전학습 모델로 객체검출을 수행합니다.

> 📌 **첫 실행 시 `yolov8n.pt` (~6MB) 가 자동 다운로드됩니다. 인터넷 연결이 필요합니다.**

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# ── 경로 설정 ──────────────────────────────────────────────
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists() and (ROOT.parent / 'scripts').exists():
    ROOT = ROOT.parent

IMAGE_DIR  = ROOT / 'data' / 'images'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
image_candidates = sorted([p for p in IMAGE_DIR.iterdir()
                            if p.suffix.lower() in image_exts]) \
                   if IMAGE_DIR.exists() else []

if not image_candidates:
    raise FileNotFoundError(
        f"이미지 파일이 없습니다.\n  → {IMAGE_DIR} 에 .jpg/.png 파일을 넣어주세요.\n"
        "  Google Colab 수강생: 상단 셀에서 !wget 또는 파일 업로드 후 재실행하세요."
    )

image_path = image_candidates[0]
print('사용 이미지:', image_path)

In [ ]:
# 모델 로드 및 객체검출 실행
model = YOLO('yolov8n.pt')
results = model.predict(
    source=str(image_path),
    conf=0.25,
    save=True,
    project=str(OUTPUT_DIR),
    name='detect_single_nb',
    exist_ok=True,
)
result = results[0]
print('검출 개수:', len(result.boxes))

In [ ]:
# 검출 결과 텍스트 출력
names = result.names
for idx, box in enumerate(result.boxes[:10]):
    class_id = int(box.cls.item())
    score    = float(box.conf.item())
    print(f'{idx + 1}. {names[class_id]} ({score:.3f})')

In [ ]:
# 결과 이미지 시각화
result_image_path = OUTPUT_DIR / 'detect_single_nb' / image_path.name
img = Image.open(result_image_path)
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.title(f'객체검출 결과 (conf=0.25)')
plt.show()

## confidence 값 비교 실습

- **낮은 conf (0.2)**: 검출 많음, 오검출(False Positive) 증가
- **중간 conf (0.4~0.5)**: 균형
- **높은 conf (0.7+)**: 검출 적음, 미검출(False Negative) 증가

In [ ]:
for conf_value in [0.2, 0.4, 0.6, 0.8]:
    r = model.predict(source=str(image_path), conf=conf_value, save=False)
    print(f'conf={conf_value}: 검출 개수 = {len(r[0].boxes)}')